# CIFAR-10 Interpretable CNNs — Formal XAI Evaluation Metrics

## Setup and Dependencies

In [1]:
import subprocess, sys
def _pip(pkg, import_name=None):
    try:
        __import__(import_name or pkg.replace('-', '_'))
        print(f'  ✓ {pkg}')
    except ImportError:
        print(f'  installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f'  ✓ {pkg}')
_pip('scikit-image', 'skimage')
_pip('seaborn')
_pip('pandas')
_pip('scipy')


  ✓ scikit-image
  ✓ seaborn
  ✓ pandas
  ✓ scipy


In [2]:
import os, copy, random, warnings, glob
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import torchvision.models as tv_models
from torch.utils.data import DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy.stats import spearmanr, wilcoxon, pearsonr
from skimage.metrics import structural_similarity as skimage_ssim
plt.style.use('seaborn-v0_8-paper')
OUT = Path('outputs/xai_metrics')
for sub in ['', '/figures', '/latex']:
    Path(str(OUT) + sub).mkdir(parents=True, exist_ok=True)
print('Imports successful.')
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Imports successful.
PyTorch 2.5.1+cu121 | CUDA: True
GPU: NVIDIA RTX 5000 Ada Generation


In [3]:
CFG = {
    'seed': 42,
    'data': {
        'root':        './data',
        'num_workers': 0,
        'mean': [0.4914, 0.4822, 0.4465],
        'std':  [0.2470, 0.2435, 0.2616],
    },
    'training': {
        'epochs': 100, 'batch_size': 128, 'lr': 0.01,
        'momentum': 0.9, 'weight_decay': 5e-4,
    },
    'models': {'save_dir': './models', 'baseline_cnn': {'dropout': 0.5}},
}
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']
EXPECTED_ACC = {
    'baseline_cnn': 87.3,
    'resnet18_scratch': 93.3,
    'resnet18_pretrained': 95.9,
}
MODEL_LABELS = {
    'baseline_cnn': 'BaselineCNN',
    'resnet18_scratch': 'ResNet18-scratch',
    'resnet18_pretrained': 'ResNet18-pretrained',
}
def set_seed(s=42):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
set_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


In [4]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.5):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(3,  32, 3, padding=1), nn.BatchNorm2d(32),
            nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),
            nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.layer3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128),
            nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128*4*4, 256),
            nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(256, num_classes))
    def forward(self, x):
        return self.classifier(self.layer3(self.layer2(self.layer1(x))))


def build_resnet18(pretrained=True, num_classes=10):
    weights = tv_models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model   = tv_models.resnet18(weights=weights)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model


def build_model(model_name, cfg):
    if model_name == 'baseline_cnn':
        return BaselineCNN(dropout=cfg['models']['baseline_cnn']['dropout'])
    elif model_name == 'resnet18_pretrained':
        return build_resnet18(pretrained=True)
    elif model_name == 'resnet18_scratch':
        return build_resnet18(pretrained=False)
    else:
        raise ValueError(f'Unknown model: {model_name}')

In [5]:
MEAN = CFG['data']['mean']
STD  = CFG['data']['std']

def denormalize(tensor, mean=MEAN, std=STD):
    m = torch.tensor(mean).view(3,1,1)
    s = torch.tensor(std).view(3,1,1)
    img = (tensor.cpu().float() * s + m).clamp(0,1)
    return (img.permute(1,2,0).numpy() * 255).astype(np.uint8)


def overlay_heatmap(img_np, cam, alpha=0.4):
    heatmap = (cm.jet(cam)[:,:,:3] * 255).astype(np.uint8)
    blended = ((1-alpha) * img_np.astype(np.float32)/255
               + alpha   * heatmap.astype(np.float32)/255)
    return (blended * 255).astype(np.uint8)


def get_target_layer(model, model_name):
    if 'resnet18' in model_name:
        return model.layer4[-1]
    elif 'baseline_cnn' in model_name or 'dropout' in model_name:
        return model.layer3
    raise ValueError(f'Unknown target layer for: {model_name}')

test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])


In [6]:
class GradCAM:


    def __init__(self, model, target_layer):
        self.model        = model
        self.target_layer = target_layer
        self._fmaps = self._grads = None
        self._fwd = target_layer.register_forward_hook(
            lambda m, i, o: setattr(self, '_fmaps', o.detach()))
        self._bwd = target_layer.register_full_backward_hook(
            lambda m, gi, go: setattr(self, '_grads', go[0].detach()))

    def __call__(self, inp, target_class=None, size=(32, 32)):
        self.model.eval()
        self.model.zero_grad()
        logits = self.model(inp)
        if target_class is None:
            target_class = logits.argmax(1).item()
        logits[0, target_class].backward()
        weights = self._grads.mean(dim=(2, 3))
        cam = (weights[:,:,None,None] * self._fmaps).sum(dim=1).squeeze(0)
        cam = F.relu(cam)
        cam = F.interpolate(cam[None,None], size=size,
                            mode='bilinear', align_corners=False).squeeze().cpu().numpy()
        lo, hi = cam.min(), cam.max()
        return (cam - lo) / (hi - lo + 1e-8)

    def remove(self):
        self._fwd.remove()
        self._bwd.remove()

    def __enter__(self): return self
    def __exit__(self, *_): self.remove()


##  Load Checkpoints and Build Image Pool

In [7]:

CKPT_PATHS = {
    'baseline_cnn':        ['./models/baseline_cnn.pth',
                            './baseline_cnn.pth',
                            './models/baseline_cnn_best.pth'],
    'resnet18_scratch':    ['./models/resnet18_scratch.pth',
                            './resnet18_scratch.pth',
                            './models/resnet18_scratch_best.pth'],
    'resnet18_pretrained': ['./models/resnet18_pretrained.pth',
                            './resnet18_pretrained.pth',
                            './models/resnet18_pretrained_best.pth'],
}

def _load_ckpt(model, model_name, device):
    for p in CKPT_PATHS[model_name]:
        if not os.path.exists(p):
            continue
        state = torch.load(p, map_location=device)
        sd = state['model_state_dict'] if 'model_state_dict' in state else state
        model.load_state_dict(sd)
        print(f'  Loaded {model_name} <- {p}')
        return
    raise FileNotFoundError(
        f'No checkpoint for {model_name}. Run training notebook first.')

@torch.no_grad()
def evaluate_accuracy(model, loader, device):
    model.eval()
    correct = total = 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        correct += (model(imgs).argmax(1) == lbls).sum().item()
        total   += lbls.size(0)
    return 100.0 * correct / total

test_dataset = torchvision.datasets.CIFAR10(
    root=CFG['data']['root'], train=False, download=True,
    transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False,
                         num_workers=0, pin_memory=True)

LOADED_MODELS = {}
print('Loading checkpoints...')
for mname in ['baseline_cnn', 'resnet18_scratch', 'resnet18_pretrained']:
    m = build_model(mname, CFG).to(DEVICE)
    _load_ckpt(m, mname, DEVICE)
    m.eval()
    acc = evaluate_accuracy(m, test_loader, DEVICE)
    exp = EXPECTED_ACC[mname]
    diff = abs(acc - exp)
    status = '✓' if diff <= 0.5 else '✗ WARN'
    print(f'  {status} {mname}: {acc:.2f}% (expected {exp:.1f}%, diff={diff:.2f}pp)')
    LOADED_MODELS[mname] = m

Files already downloaded and verified
Loading checkpoints...
  Loaded baseline_cnn <- ./models/baseline_cnn_best.pth


C:\Users\User 1\AppData\Local\Temp\ipykernel_44840\3806488338.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(p, map_location=device)


  ✓ baseline_cnn: 87.10% (expected 87.3%, diff=0.20pp)
  Loaded resnet18_scratch <- ./models/resnet18_scratch_best.pth
  ✓ resnet18_scratch: 92.93% (expected 93.3%, diff=0.37pp)
  Loaded resnet18_pretrained <- ./models/resnet18_pretrained_best.pth
  ✓ resnet18_pretrained: 96.15% (expected 95.9%, diff=0.25pp)


In [8]:
N_PER_CLASS = 5

EVAL_IMAGES = {i: [] for i in range(10)}
for idx in range(len(test_dataset)):
    img_t, label = test_dataset[idx]
    if len(EVAL_IMAGES[label]) >= N_PER_CLASS:
        continue
    with torch.no_grad():
        inp = img_t.unsqueeze(0).to(DEVICE)
        if all(m(inp).argmax(1).item() == label for m in LOADED_MODELS.values()):
            EVAL_IMAGES[label].append(img_t)
    if all(len(v) == N_PER_CLASS for v in EVAL_IMAGES.values()):
        break
total = sum(len(v) for v in EVAL_IMAGES.values())
print(f'Selected {total} images ({N_PER_CLASS} per class x 10 classes):')
for cls_idx in range(10):
    n = len(EVAL_IMAGES[cls_idx])
    status = '✓' if n == N_PER_CLASS else f'✗ only {n}'
    print(f'  [{cls_idx}] {CLASSES[cls_idx]:<12}: {status}')

assert total == 50, f'Expected 50 images, got {total}. Increase test set search or N_PER_CLASS.'

Selected 50 images (5 per class x 10 classes):
  [0] airplane    : ✓
  [1] automobile  : ✓
  [2] bird        : ✓
  [3] cat         : ✓
  [4] deer        : ✓
  [5] dog         : ✓
  [6] frog        : ✓
  [7] horse       : ✓
  [8] ship        : ✓
  [9] truck       : ✓


In [9]:
ALL_CAMS = {mname: {i: [] for i in range(10)} for mname in LOADED_MODELS}

for mname, model in LOADED_MODELS.items():
    model.eval()
    target_layer = get_target_layer(model, mname)
    gcam_ctx = GradCAM(model, target_layer)
    for cls_idx in range(10):
        for img_t in EVAL_IMAGES[cls_idx]:
            inp = img_t.unsqueeze(0).to(DEVICE)
            cam = gcam_ctx(inp, target_class=cls_idx)
            ALL_CAMS[mname][cls_idx].append(cam)
    gcam_ctx.remove()
    print(f'  ✓ {mname}: {10 * N_PER_CLASS} heatmaps')

print(f'\nTotal heatmaps: {3 * 50}')

  ✓ baseline_cnn: 50 heatmaps
  ✓ resnet18_scratch: 50 heatmaps
  ✓ resnet18_pretrained: 50 heatmaps

Total heatmaps: 150


##  Formal Metric Implementations


In [10]:
def ssim_scratch(x: np.ndarray, y: np.ndarray,
                 K1: float = 0.01, K2: float = 0.03,
                 win_size: int = 7, data_range: float = 1.0) -> float:
    if x.shape != y.shape:
        raise ValueError(f'Shape mismatch: {x.shape} vs {y.shape}')
    if x.size == 1:
        return 1.0 if np.isclose(x.flat[0], y.flat[0]) else 0.0

    x = x.astype(np.float64)
    y = y.astype(np.float64)

    C1 = (K1 * data_range) ** 2
    C2 = (K2 * data_range) ** 2
    C3 = C2 / 2.0

    H, W = x.shape
    half = win_size // 2

    ssim_vals = []
    for i in range(half, H - half):
        for j in range(half, W - half):
            px = x[i-half:i+half+1, j-half:j+half+1].ravel()
            py = y[i-half:i+half+1, j-half:j+half+1].ravel()

            mu_x   = px.mean()
            mu_y   = py.mean()
            sig_x  = px.std(ddof=0)
            sig_y  = py.std(ddof=0)
            sig_xy = np.mean((px - mu_x) * (py - mu_y))

            l = (2*mu_x*mu_y + C1) / (mu_x**2 + mu_y**2 + C1)
            c = (2*sig_x*sig_y + C2) / (sig_x**2 + sig_y**2 + C2)
            s = (sig_xy + C3) / (sig_x*sig_y + C3)

            ssim_vals.append(l * c * s)

    return float(np.mean(ssim_vals)) if ssim_vals else 0.0


def ssim(x: np.ndarray, y: np.ndarray) -> float:
    return float(skimage_ssim(x, y, data_range=1.0, win_size=7))

np.random.seed(42)
test_pairs = [(np.random.rand(32,32), np.random.rand(32,32)) for _ in range(50)]

ssim_sk     = np.array([ssim(a, b)         for a, b in test_pairs])
ssim_manual = np.array([ssim_scratch(a, b) for a, b in test_pairs])

r, p = pearsonr(ssim_sk, ssim_manual)
print(f'  Pearson r = {r:.6f}  (p = {p:.2e})')
assert r > 0.99, f'SSIM implementations diverge: r={r:.4f} < 0.99'


  Pearson r = 1.000000  (p = 1.98e-200)


In [11]:
def compute_lcs(heatmaps: list) -> float:
    N = len(heatmaps)
    if N < 2:
        return float('nan')
    pairwise = [ssim(heatmaps[i], heatmaps[j])
                for i, j in combinations(range(N), 2)]
    return float(np.mean(pairwise))

ones = [np.ones((32,32)) for _ in range(5)]
print(f'  LCS(5x identical) = {compute_lcs(ones):.4f}  (expected 1.0)')
rand_maps = [np.random.rand(32,32) for _ in range(5)]
print(f'  LCS(5x random)    = {compute_lcs(rand_maps):.4f}  (expected ~low)')

  LCS(5x identical) = 1.0000  (expected 1.0)
  LCS(5x random)    = 0.0071  (expected ~low)


In [12]:
def get_layer_groups(model, model_name):
    if 'resnet18' in model_name:
        return [
            ('fc',     model.fc),
            ('layer4', model.layer4),
            ('layer3', model.layer3),
            ('layer2', model.layer2),
            ('layer1', model.layer1),
        ]
    else:
        return [
            ('classifier', model.classifier),
            ('layer3',     model.layer3),
            ('layer2',     model.layer2),
            ('layer1',     model.layer1),
        ]


def randomize_group(layer_group):
    """Re-initialize all parameters in a layer group with N(0,1)."""
    for mod in (layer_group.modules()
                if hasattr(layer_group, 'modules') else [layer_group]):
        for p in mod.parameters(recurse=False):
            nn.init.normal_(p)


def compute_rs_curve(model, model_name, heatmaps_orig: list,
                     eval_images: list, device, n_stages: int = 5):
    model_copy   = copy.deepcopy(model)
    model_copy.eval()
    layer_groups = get_layer_groups(model_copy, model_name)
    n_stages     = min(n_stages, len(layer_groups))

    rs_means, rs_stds = [], []

    for stage in range(1, n_stages + 1):
        _, lgrp = layer_groups[stage - 1]
        randomize_group(lgrp)

        target_layer = get_target_layer(model_copy, model_name)
        gcam_ctx = GradCAM(model_copy, target_layer)

        stage_rs = []
        for img_t, h0 in zip(eval_images, heatmaps_orig):
            inp  = img_t.unsqueeze(0).to(device)
            hk   = gcam_ctx(inp, target_class=None)
            rs_k = 1.0 - ssim(h0, hk)
            stage_rs.append(rs_k)
        gcam_ctx.remove()

        rs_means.append(float(np.mean(stage_rs)))
        rs_stds.append(float(np.std(stage_rs)))

    del model_copy
    rs_means = np.array(rs_means)
    rs_stds  = np.array(rs_stds)

    monotonic = bool(np.all(np.diff(rs_means) > 0))

    return rs_means, rs_stds, monotonic


In [13]:
def compute_eec(heatmap: np.ndarray, p: float = 25.0) -> float:
    h = heatmap.astype(np.float64).ravel()
    total = h.sum()
    if total < 1e-10:
        return 0.0   # All-zero heatmap — undefined; return 0 by convention

    h_norm = h / total
    n_top  = max(1, int(np.ceil(len(h) * p / 100.0)))
    top_indices = np.argpartition(h_norm, -n_top)[-n_top:]
    return float(h_norm[top_indices].sum())


uniform = np.ones((32,32))
print(f'  EEC(uniform, p=25) = {compute_eec(uniform):.4f}  (expected 0.25)')
spike   = np.zeros((32,32)); spike[16,16] = 1.0
print(f'  EEC(spike,   p=25) = {compute_eec(spike):.4f}   (expected ~1.0, only 1 non-zero pixel)')
zeros   = np.zeros((32,32))
print(f'  EEC(zeros,   p=25) = {compute_eec(zeros):.4f}   (expected 0.0)')

  EEC(uniform, p=25) = 0.2500  (expected 0.25)
  EEC(spike,   p=25) = 1.0000   (expected ~1.0, only 1 non-zero pixel)
  EEC(zeros,   p=25) = 0.0000   (expected 0.0)


In [14]:
def compute_icd(cams_per_class: dict) -> float:
    keys = sorted(cams_per_class.keys())
    if len(keys) < 2:
        return float('nan')
    heatmaps = [cams_per_class[k] for k in keys]
    pairwise = [ssim(heatmaps[i], heatmaps[j])
                for i, j in combinations(range(len(heatmaps)), 2)]
    return float(1.0 - np.mean(pairwise))
all_same   = {i: np.ones((32,32)) for i in range(10)}
all_diff   = {i: (np.eye(32) * i % 1) for i in range(10)}
print(f'  ICD(all identical heatmaps) = {compute_icd(all_same):.4f}  (expected 0.0)')
rnd_per_cls = {i: np.random.rand(32,32) for i in range(10)}
print(f'  ICD(random per-class maps)  = {compute_icd(rnd_per_cls):.4f}  (expected >0)')

  ICD(all identical heatmaps) = 0.0000  (expected 0.0)
  ICD(random per-class maps)  = 0.9953  (expected >0)


##  Systematic Metric Computation

In [15]:
LCS_TABLE = {}

for mname in LOADED_MODELS:
    LCS_TABLE[mname] = {}
    for cls_idx in range(10):
        hms = ALL_CAMS[mname][cls_idx]
        LCS_TABLE[mname][cls_idx] = compute_lcs(hms)
lcs_df = pd.DataFrame(
    {MODEL_LABELS[mn]: [LCS_TABLE[mn][i] for i in range(10)]
     for mn in LOADED_MODELS},
    index=CLASSES
)
lcs_df.index.name = 'Class'
lcs_df.loc['MEAN'] = lcs_df.mean()

print(lcs_df.round(4).to_string())

            BaselineCNN  ResNet18-scratch  ResNet18-pretrained
Class                                                         
airplane         0.0829            0.8133               0.6734
automobile       0.2815            0.7204               0.7848
bird             0.1115            0.6348               0.7650
cat              0.2406            0.7552               0.8425
deer             0.1213            0.5796               0.7671
dog              0.1340            0.7213               0.9133
frog             0.0810            0.7703               0.7546
horse            0.1436            0.7804               0.7556
ship             0.0401            0.7169               0.8151
truck            0.2392            0.8013               0.8671
MEAN             0.1476            0.7293               0.7938


### 2b — EEC per model (mean ± std)

In [16]:
EEC_ALL = {mname: [] for mname in LOADED_MODELS}

for mname in LOADED_MODELS:
    for cls_idx in range(10):
        for cam in ALL_CAMS[mname][cls_idx]:
            EEC_ALL[mname].append(compute_eec(cam, p=25))

eec_summary = {MODEL_LABELS[mn]: {
    'mean': float(np.mean(EEC_ALL[mn])),
    'std':  float(np.std(EEC_ALL[mn])),
} for mn in LOADED_MODELS}
for lbl, v in eec_summary.items():
    print(f'  {lbl:<25}: {v["mean"]:.4f} ± {v["std"]:.4f}')

  BaselineCNN              : 0.4836 ± 0.0765
  ResNet18-scratch         : 0.4261 ± 0.0261
  ResNet18-pretrained      : 0.4197 ± 0.0175


### 2c — EEC for dropout ablation variants

In [17]:


DROPOUT_RATES = [0.0, 0.2, 0.4, 0.6, 0.8]

EEC_DROPOUT = {}

for dr in DROPOUT_RATES:
    dr_str = str(dr).replace('.', '')
    ckpt_p = f'models/baseline_dropout_{dr_str}.pth'
    if not os.path.exists(ckpt_p):
        print(f'  [SKIP] {ckpt_p} not found (run notebook 2 §3 first)')
        EEC_DROPOUT[dr] = None
        continue
    model = BaselineCNN(dropout=dr).to(DEVICE)
    state = torch.load(ckpt_p, map_location=DEVICE)
    sd = state['model_state_dict'] if 'model_state_dict' in state else state
    model.load_state_dict(sd)
    model.eval()
    gcam_ctx = GradCAM(model, model.layer3)
    eec_vals = []
    for cls_idx in range(10):
        for img_t in EVAL_IMAGES[cls_idx]:
            cam = gcam_ctx(img_t.unsqueeze(0).to(DEVICE), target_class=cls_idx)
            eec_vals.append(compute_eec(cam, p=25))
    gcam_ctx.remove()
    EEC_DROPOUT[dr] = np.array(eec_vals)
    print(f'  dr={dr}: EEC mean={np.mean(eec_vals):.4f} ± {np.std(eec_vals):.4f}')


C:\Users\User 1\AppData\Local\Temp\ipykernel_44840\3434697083.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt_p, map_location=DEVICE)


  dr=0.0: EEC mean=0.4956 ± 0.0866
  dr=0.2: EEC mean=0.5022 ± 0.0966
  dr=0.4: EEC mean=0.4938 ± 0.0691
  dr=0.6: EEC mean=0.4978 ± 0.0778
  dr=0.8: EEC mean=0.5267 ± 0.0908


### 2d — ICD per model

In [18]:
ICD_SCORES = {}
for mname in LOADED_MODELS:
    # Use first image per class as representative
    repr_cams = {cls_idx: ALL_CAMS[mname][cls_idx][0] for cls_idx in range(10)}
    ICD_SCORES[mname] = compute_icd(repr_cams)
for mn, score in ICD_SCORES.items():
    print(f'  {MODEL_LABELS[mn]:<25}: {score:.4f}')

  BaselineCNN              : 0.8452
  ResNet18-scratch         : 0.2852
  ResNet18-pretrained      : 0.1827


### 2e — RS curves and monotonicity check per model

In [19]:
RS_MEANS     = {}
RS_STDS      = {}
RS_MONOTONIC = {}
for mname, model in LOADED_MODELS.items():
    print(f'\nComputing RS — {mname}...')

    flat_imgs = []
    flat_cams = []
    for cls_idx in range(10):
        for k, img_t in enumerate(EVAL_IMAGES[cls_idx]):
            flat_imgs.append(img_t)
            flat_cams.append(ALL_CAMS[mname][cls_idx][k])

    n_stages = 5 if 'resnet18' in mname else 4

    rs_m, rs_s, mono = compute_rs_curve(
        model, mname, flat_cams, flat_imgs, DEVICE, n_stages=n_stages)

    RS_MEANS[mname]     = rs_m
    RS_STDS[mname]      = rs_s
    RS_MONOTONIC[mname] = mono
    status = 'PASS' if mono else 'FAIL'
    print(f'  Monotonicity: {status}')
    for k, (m, s) in enumerate(zip(rs_m, rs_s), 1):
        print(f'  RS_{k} = {m:.4f} ± {s:.4f}')


Computing RS — baseline_cnn...
  Monotonicity: PASS
  RS_1 = 0.8237 ± 0.2201
  RS_2 = 0.8942 ± 0.2006
  RS_3 = 0.9472 ± 0.2048
  RS_4 = 0.9729 ± 0.2093

Computing RS — resnet18_scratch...
  Monotonicity: FAIL
  RS_1 = 0.5914 ± 0.2513
  RS_2 = 0.9943 ± 0.1496
  RS_3 = 0.9236 ± 0.1418
  RS_4 = 1.1033 ± 0.0830
  RS_5 = 1.0427 ± 0.1265

Computing RS — resnet18_pretrained...
  Monotonicity: FAIL
  RS_1 = 0.2022 ± 0.1733
  RS_2 = 0.7287 ± 0.1874
  RS_3 = 0.5444 ± 0.1584
  RS_4 = 0.5887 ± 0.1467
  RS_5 = nan ± nan


## Section 3 — Statistical Validation and Figures

In [20]:

model_names = list(LOADED_MODELS.keys())

def sig_label(p):
    if p < 0.01: return '**'
    if p < 0.05: return '*'
    return 'ns'
for i in range(len(model_names)):
    for j in range(i+1, len(model_names)):
        mn_a, mn_b = model_names[i], model_names[j]
        lcs_a = np.array([LCS_TABLE[mn_a][c] for c in range(10)])
        lcs_b = np.array([LCS_TABLE[mn_b][c] for c in range(10)])
        if np.allclose(lcs_a, lcs_b):
            print(f'  {MODEL_LABELS[mn_a]} vs {MODEL_LABELS[mn_b]}: identical (skip)')
            continue
        _, p = wilcoxon(lcs_a, lcs_b)
        print(f'  {MODEL_LABELS[mn_a]} vs {MODEL_LABELS[mn_b]}: p={p:.4f} {sig_label(p)}')
for i in range(len(model_names)):
    for j in range(i+1, len(model_names)):
        mn_a, mn_b = model_names[i], model_names[j]
        eec_a = np.array(EEC_ALL[mn_a])
        eec_b = np.array(EEC_ALL[mn_b])
        if np.allclose(eec_a, eec_b):
            continue
        _, p = wilcoxon(eec_a, eec_b)
        print(f'  {MODEL_LABELS[mn_a]} vs {MODEL_LABELS[mn_b]}: p={p:.4f} {sig_label(p)}')

  BaselineCNN vs ResNet18-scratch: p=0.0020 **
  BaselineCNN vs ResNet18-pretrained: p=0.0020 **
  ResNet18-scratch vs ResNet18-pretrained: p=0.1055 ns
  BaselineCNN vs ResNet18-scratch: p=0.0000 **
  BaselineCNN vs ResNet18-pretrained: p=0.0000 **
  ResNet18-scratch vs ResNet18-pretrained: p=0.1689 ns


In [21]:

lcs_plot_df = lcs_df.drop(index='MEAN')

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    lcs_plot_df.astype(float),
    ax=ax,
    cmap='YlOrRd',
    vmin=0, vmax=1,
    annot=True,
    fmt='.2f',
    linewidths=0.5,
    cbar_kws={'label': 'LCS (mean pairwise SSIM)'},
)
ax.set_title('Localization Consistency Score (LCS) — per Class per Model',
             fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Model', fontsize=11)
ax.set_ylabel('CIFAR-10 Class', fontsize=11)
ax.tick_params(axis='x', rotation=15)
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.savefig(str(OUT / 'figures/lcs_heatmap.png'), dpi=150, bbox_inches='tight')
plt.close()


In [22]:
fig, ax = plt.subplots(figsize=(8, 6))
x_pos = np.arange(len(model_names))
bar_colors = ['#3B82F6', '#10B981', '#F59E0B']
for xi, (mname, color) in enumerate(zip(model_names, bar_colors)):
    vals = np.array(EEC_ALL[mname])
    ax.bar(xi, vals.mean(), width=0.5, color=color, alpha=0.75,
           label=MODEL_LABELS[mname], zorder=2)
    ax.errorbar(xi, vals.mean(), yerr=vals.std(), fmt='none',
                color='black', capsize=5, lw=1.5, zorder=3)
    jitter = np.random.uniform(-0.18, 0.18, size=len(vals))
    ax.scatter(xi + jitter, vals, color=color, s=15, alpha=0.45, zorder=4,
               edgecolors='none')
ax.set_xticks(x_pos)
ax.set_xticklabels([MODEL_LABELS[mn] for mn in model_names], fontsize=10)
ax.set_ylabel('EEC (p=25%)', fontsize=11)
ax.set_title('Energy-Based Explanation Concentration (EEC) per Model\n'
             'bars=mean±std; dots=individual images', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3, zorder=1)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(str(OUT / 'figures/eec_comparison.png'), dpi=150, bbox_inches='tight')
plt.close()
print('Saved: outputs/xai_metrics/figures/eec_comparison.png')

Saved: outputs/xai_metrics/figures/eec_comparison.png


In [23]:
fig, ax = plt.subplots(figsize=(7, 5))
icd_vals  = [ICD_SCORES[mn] for mn in model_names]
bar_colors = ['#3B82F6', '#10B981', '#F59E0B']
bars = ax.bar([MODEL_LABELS[mn] for mn in model_names],
              icd_vals, color=bar_colors, width=0.5, alpha=0.85, edgecolor='black', lw=0.5)
for bar, v in zip(bars, icd_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{v:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('ICD', fontsize=11)
ax.set_title('Inter-Class Discriminability (ICD) per Model', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.tick_params(axis='x', rotation=12)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(str(OUT / 'figures/icd_comparison.png'), dpi=150, bbox_inches='tight')
plt.close()

In [24]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
fig.suptitle('Randomization Sensitivity (RS) Curves — All Models',
             fontsize=13, fontweight='bold')
for ax, (mname, color) in zip(axes, zip(model_names, ['#3B82F6','#10B981','#F59E0B'])):
    rs_m = RS_MEANS[mname]
    rs_s = RS_STDS[mname]
    stages = np.arange(1, len(rs_m) + 1)
    x_full = np.concatenate([[0], stages])
    y_full = np.concatenate([[0.0], rs_m])
    y_lo   = np.concatenate([[0.0], rs_m - rs_s])
    y_hi   = np.concatenate([[0.0], rs_m + rs_s])
    ax.plot(x_full, y_full, 'o-', color=color, lw=2, ms=8)
    ax.fill_between(x_full, y_lo, y_hi, alpha=0.2, color=color, label='±1 std')
    ax.axhline(0.5, color='#DC2626', ls='--', lw=1, alpha=0.7, label='RS=0.5')
    mono_str = '✓ Monotonic' if RS_MONOTONIC[mname] else '✗ Non-monotonic'
    ax.set_title(f'{MODEL_LABELS[mname]}\n{mono_str}', fontsize=10)
    ax.set_xlabel('Cascade Stage (0=trained)')
    ax.set_xticks(x_full)
    ax.set_xticklabels(['Original'] + [f'S{k}' for k in stages], rotation=10)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
axes[0].set_ylabel('RS_k = 1 - SSIM(H_0, H_k)', fontsize=10)
plt.tight_layout()
plt.savefig(str(OUT / 'figures/rs_curves.png'), dpi=150, bbox_inches='tight')
plt.close()

In [25]:
available_drs = [dr for dr in DROPOUT_RATES if EEC_DROPOUT.get(dr) is not None]

if len(available_drs) >= 2:
    eec_dr_means = [EEC_DROPOUT[dr].mean() for dr in available_drs]
    eec_dr_stds  = [EEC_DROPOUT[dr].std()  for dr in available_drs]
    eec_orig_mean = eec_summary[MODEL_LABELS['baseline_cnn']]['mean']
    eec_orig_std  = eec_summary[MODEL_LABELS['baseline_cnn']]['std']

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.errorbar(available_drs, eec_dr_means, yerr=eec_dr_stds,
                fmt='o-', color='#2563EB', lw=2, ms=8, capsize=5, label='Ablated')
    ax.errorbar([0.5], [eec_orig_mean], yerr=[eec_orig_std],
                fmt='s', color='#DC2626', ms=10, capsize=5,
                label=f'Original (dr=0.5, EEC={eec_orig_mean:.3f})')
    ax.set_xlabel('Dropout Rate', fontsize=12)
    ax.set_ylabel('Mean EEC (p=25%)', fontsize=12)
    ax.set_title('EEC vs Dropout Rate — BaselineCNN', fontsize=12, fontweight='bold')
    ax.set_xticks(sorted(set(available_drs + [0.5])))
    ax.set_ylim(0, 1.0)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(OUT / 'figures/eec_dropout.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print('Saved: outputs/xai_metrics/figures/eec_dropout.png')
else:
    print('[SKIP] Not enough dropout checkpoints available for EEC vs dropout plot.')

Saved: outputs/xai_metrics/figures/eec_dropout.png


## LaTeX Table Generation

---



In [26]:
def bold(val_str, is_best):
    return r'\textbf{' + val_str + '}' if is_best else val_str
lcs_means = {mn: np.mean([LCS_TABLE[mn][c] for c in range(10)])
             for mn in model_names}
lcs_stds  = {mn: np.std([LCS_TABLE[mn][c] for c in range(10)])
             for mn in model_names}
eec_means = {mn: np.mean(EEC_ALL[mn]) for mn in model_names}
eec_stds  = {mn: np.std(EEC_ALL[mn]) for mn in model_names}

best_lcs = max(model_names, key=lambda mn: lcs_means[mn])
best_eec = max(model_names, key=lambda mn: eec_means[mn])
best_icd = max(model_names, key=lambda mn: ICD_SCORES[mn])
latex_lines = []
latex_lines.append(r'% ── Table 1: Quantitative Interpretability Metrics ──────────────────')
latex_lines.append(r'\begin{table}[h]')
latex_lines.append(r'\centering')
latex_lines.append(r'\caption{Quantitative Interpretability Metrics for Grad-CAM Across Three CIFAR-10 Classifiers}')
latex_lines.append(r'\label{tab:xai_metrics}')
latex_lines.append(r'\begin{tabular}{lcccc}')
latex_lines.append(r'\hline')
latex_lines.append(r'\textbf{Model} & \textbf{LCS (mean$\pm$std)} & \textbf{EEC (mean$\pm$std)} & \textbf{ICD} & \textbf{RS Monotonic?} \\')
latex_lines.append(r'\hline')
for mn in model_names:
    lbl    = MODEL_LABELS[mn]
    lcs_s  = f'{lcs_means[mn]:.3f}$\\pm${lcs_stds[mn]:.3f}'
    eec_s  = f'{eec_means[mn]:.3f}$\\pm${eec_stds[mn]:.3f}'
    icd_s  = f'{ICD_SCORES[mn]:.3f}'
    mono_s = 'Yes' if RS_MONOTONIC[mn] else 'No'
    lcs_s  = bold(lcs_s, mn == best_lcs)
    eec_s  = bold(eec_s, mn == best_eec)
    icd_s  = bold(icd_s, mn == best_icd)
    latex_lines.append(f'{lbl} & {lcs_s} & {eec_s} & {icd_s} & {mono_s} \\\\')
latex_lines.append(r'\hline')
latex_lines.append(r'\end{tabular}')
latex_lines.append(r'\end{table}')
table1 = '\n'.join(latex_lines)
print(table1)
with open(str(OUT / 'latex/table1_summary.tex'), 'w', encoding='utf-8') as f:
    f.write(table1)

% ── Table 1: Quantitative Interpretability Metrics ──────────────────
\begin{table}[h]
\centering
\caption{Quantitative Interpretability Metrics for Grad-CAM Across Three CIFAR-10 Classifiers}
\label{tab:xai_metrics}
\begin{tabular}{lcccc}
\hline
\textbf{Model} & \textbf{LCS (mean$\pm$std)} & \textbf{EEC (mean$\pm$std)} & \textbf{ICD} & \textbf{RS Monotonic?} \\
\hline
BaselineCNN & 0.148$\pm$0.076 & \textbf{0.484$\pm$0.076} & \textbf{0.845} & Yes \\
ResNet18-scratch & 0.729$\pm$0.070 & 0.426$\pm$0.026 & 0.285 & No \\
ResNet18-pretrained & \textbf{0.794$\pm$0.064} & 0.420$\pm$0.017 & 0.183 & No \\
\hline
\end{tabular}
\end{table}


In [31]:
latex_lines = []
latex_lines.append(r'% Table 2: Per-Class LCS Breakdown')
latex_lines.append(r'\begin{table}[h]')
latex_lines.append(r'\centering')
latex_lines.append(r'\caption{Per-Class Localization Consistency Score (LCS) — Mean Pairwise SSIM across 5 Images}')
latex_lines.append(r'\label{tab:lcs_per_class}')
latex_lines.append(r'\begin{tabular}{lccc}')
latex_lines.append(r'\hline')
latex_lines.append(
    r'\textbf{Class} & \textbf{BaselineCNN} & \textbf{ResNet18-scratch} & \textbf{ResNet18-pretrained} \\'
)
latex_lines.append(r'\hline')

for cls_idx, cls_name in enumerate(CLASSES):
    row_vals = {mn: LCS_TABLE[mn][cls_idx] for mn in model_names}
    best_mn  = max(row_vals, key=row_vals.get)
    cells    = []
    for mn in model_names:
        v = f'{row_vals[mn]:.3f}'
        cells.append(bold(v, mn == best_mn))
    latex_lines.append(f'{cls_name.capitalize()} & ' + ' & '.join(cells) + r' \\')
latex_lines.append(r'\hline')
mean_cells = []
for mn in model_names:
    v    = f'{lcs_means[mn]:.3f}'
    mean_cells.append(bold(v, mn == best_lcs))
latex_lines.append(r'\textbf{Mean} & ' + ' & '.join(mean_cells) + r' \\')
latex_lines.append(r'\hline')
latex_lines.append(r'\end{tabular}')
latex_lines.append(r'\end{table}')
table2 = '\n'.join(latex_lines)
print(table2)
with open(str(OUT / 'latex/table2_lcs_per_class.tex'), 'w', encoding='utf-8') as f:
    f.write(table2)

% Table 2: Per-Class LCS Breakdown
\begin{table}[h]
\centering
\caption{Per-Class Localization Consistency Score (LCS) — Mean Pairwise SSIM across 5 Images}
\label{tab:lcs_per_class}
\begin{tabular}{lccc}
\hline
\textbf{Class} & \textbf{BaselineCNN} & \textbf{ResNet18-scratch} & \textbf{ResNet18-pretrained} \\
\hline
Airplane & 0.083 & \textbf{0.813} & 0.673 \\
Automobile & 0.281 & 0.720 & \textbf{0.785} \\
Bird & 0.112 & 0.635 & \textbf{0.765} \\
Cat & 0.241 & 0.755 & \textbf{0.843} \\
Deer & 0.121 & 0.580 & \textbf{0.767} \\
Dog & 0.134 & 0.721 & \textbf{0.913} \\
Frog & 0.081 & \textbf{0.770} & 0.755 \\
Horse & 0.144 & \textbf{0.780} & 0.756 \\
Ship & 0.040 & 0.717 & \textbf{0.815} \\
Truck & 0.239 & 0.801 & \textbf{0.867} \\
\hline
\textbf{Mean} & 0.148 & 0.729 & \textbf{0.794} \\
\hline
\end{tabular}
\end{table}


## Metric Justification and Limitations


## Final Results Summary

In [30]:
print('\nXAI EVALUATION METRICS — FINAL RESULTS SUMMARY')
print('\n[ LCS — Localization Consistency Score (mean ± std across 10 classes) ]')
print(f'{"Model":<28} {"LCS mean":>10} {"LCS std":>10}')
for mn in model_names:
    print(f'{MODEL_LABELS[mn]:<28} {lcs_means[mn]:>10.4f} {lcs_stds[mn]:>10.4f}')
print('\n[ EEC — Energy-Based Explanation Concentration (mean ± std, 50 images) ]')
print(f'{"Model":<28} {"EEC mean":>10} {"EEC std":>10}')
for mn in model_names:
    print(f'{MODEL_LABELS[mn]:<28} {eec_means[mn]:>10.4f} {eec_stds[mn]:>10.4f}')
print('\n[ ICD — Inter-Class Discriminability ]')
print(f'{"Model":<28} {"ICD":>8}')
for mn in model_names:
    print(f'{MODEL_LABELS[mn]:<28} {ICD_SCORES[mn]:>8.4f}')
print('\n[ RS — Randomization Sensitivity Monotonicity ]')
print(f'{"Model":<28} {"Monotonic?":>12}')
for mn in model_names:
    print(f'{MODEL_LABELS[mn]:<28} {"✓ PASS" if RS_MONOTONIC[mn] else "✗ FAIL":>12}')
print('\n[ Output Files ]')
for f in sorted(glob.glob('outputs/xai_metrics/**/*', recursive=True)):
    if os.path.isfile(f):
        print(f'  {f}')
print('All metrics computed. LaTeX tables and figures saved to outputs/xai_metrics/')


XAI EVALUATION METRICS — FINAL RESULTS SUMMARY

[ LCS — Localization Consistency Score (mean ± std across 10 classes) ]
Model                          LCS mean    LCS std
BaselineCNN                      0.1476     0.0757
ResNet18-scratch                 0.7293     0.0699
ResNet18-pretrained              0.7938     0.0645

[ EEC — Energy-Based Explanation Concentration (mean ± std, 50 images) ]
Model                          EEC mean    EEC std
BaselineCNN                      0.4836     0.0765
ResNet18-scratch                 0.4261     0.0261
ResNet18-pretrained              0.4197     0.0175

[ ICD — Inter-Class Discriminability ]
Model                             ICD
BaselineCNN                    0.8452
ResNet18-scratch               0.2852
ResNet18-pretrained            0.1827

[ RS — Randomization Sensitivity Monotonicity ]
Model                          Monotonic?
BaselineCNN                        ✓ PASS
ResNet18-scratch                   ✗ FAIL
ResNet18-pretrained           